# Notebook 62 — Static Symbolic Law as Final Coordinate Chart

Notebooks 59–61 tested whether coefficient variation should be modeled as a differential flow.
The result was decisive:

- affine latent ODEs were too straight
- quadratic latent ODEs overfit and still missed target paths
- static symbolic prediction remained much better

So Notebook 62 stops expanding latent dynamics and formalizes the current winner:

\[
\beta = f(\text{forcing}, \text{flow}, \text{system}, \text{task}, k)
\]

as the final coordinate chart for the coefficient manifold.

## Main goals

1. Fit sparse static symbolic coefficient laws.
2. Measure term stability across cross-validation folds.
3. Prune terms into a minimal symbolic chart.
4. Compare full symbolic vs pruned symbolic vs ridge.
5. Extract regime-family rules and coefficient templates.
6. Produce paper-ready equations and a final decision table.

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LassoCV, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, LeaveOneOut

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Shared governing template and coefficient table

TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "template_rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, beta):
    return design_template(sub) @ beta

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
            g = float(np.clip(x @ beta, -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        errs.append(math.sqrt(mean_squared_error(integrate(beta_ref, r0), integrate(beta_cmp, r0))))
    return float(np.mean(errs))

group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(group_cols):
    if len(sub) < 30:
        continue
    beta, pred, stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)
    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
display(coef_df.head())

## Symbolic feature chart

Feature set is intentionally readable:

- metadata dummies
- `k`, `k²`
- forcing × flow
- system × forcing
- forcing × k

In [ ]:
def build_symbolic_features(df_in, columns=None, reduced_terms=None):
    X = pd.get_dummies(df_in[["forcing_mode", "flow_mode", "system", "task"]], drop_first=False)

    X["k"] = df_in["k"].astype(float).values
    X["k2"] = df_in["k"].astype(float).values ** 2

    ff = df_in["forcing_mode"].astype(str) + "__x__" + df_in["flow_mode"].astype(str)
    X_ff = pd.get_dummies(ff, prefix="ff")

    sf = df_in["system"].astype(str) + "__x__" + df_in["forcing_mode"].astype(str)
    X_sf = pd.get_dummies(sf, prefix="sf")

    X_fk = pd.get_dummies(df_in["forcing_mode"].astype(str), prefix="fk").multiply(
        df_in["k"].astype(float).to_numpy().reshape(-1, 1)
    )

    X = pd.concat([X, X_ff, X_sf, X_fk], axis=1)

    if reduced_terms is not None:
        X = X.reindex(columns=reduced_terms, fill_value=0.0)

    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)

    return X.astype(float)

X_full = build_symbolic_features(coef_df)
print("Full symbolic feature count:", X_full.shape[1])
display(X_full.head())

## Full sparse symbolic chart

Fit one sparse model per coefficient using all regimes. These equations are descriptive, not holdout estimates.

In [ ]:
def fit_sparse_models(train_df, feature_terms=None, alpha=None):
    X = build_symbolic_features(train_df, reduced_terms=feature_terms)
    models = {}
    preds = np.zeros((len(train_df), len(COEF_COLS)), dtype=float)
    terms = {}

    for j, coef in enumerate(COEF_COLS):
        y = train_df[coef].to_numpy(dtype=float)
        scaler = StandardScaler()
        Xs = scaler.fit_transform(X)

        if alpha is None:
            model = LassoCV(cv=min(5, len(train_df)), random_state=42, max_iter=30000)
        else:
            model = Lasso(alpha=alpha, random_state=42, max_iter=30000)

        model.fit(Xs, y)
        preds[:, j] = model.predict(Xs)

        active = []
        for feat, coef_val in zip(X.columns, model.coef_):
            if abs(coef_val) > 1e-4:
                active.append((feat, float(coef_val)))

        models[coef] = {"model": model, "scaler": scaler, "columns": X.columns.tolist()}
        terms[coef] = active

    return models, preds, terms

full_models, full_preds, full_terms = fit_sparse_models(coef_df)

def format_equation(name, term_list, intercept):
    parts = [f"{name} = {intercept:.5f}"]
    for feat, val in term_list:
        sign = "+" if val >= 0 else "-"
        parts.append(f"{sign} {abs(val):.5f}·{feat}")
    return " ".join(parts)

equation_rows = []
for coef in COEF_COLS:
    m = full_models[coef]["model"]
    eq = format_equation(coef, full_terms[coef], m.intercept_)
    equation_rows.append({"coefficient": coef, "n_terms": len(full_terms[coef]), "equation": eq})
    print(eq)
    print()

equations_df = pd.DataFrame(equation_rows)
display(equations_df)

## Term stability across folds

A term is considered stable if it appears in a high fraction of held-out fits.

In [ ]:
def term_stability_table(df_in, n_splits=8, threshold=1e-4):
    n = len(df_in)
    if n <= 12:
        splitter = LeaveOneOut()
    else:
        splitter = KFold(n_splits=min(n_splits, n), shuffle=True, random_state=42)

    all_features = build_symbolic_features(df_in).columns.tolist()
    stability = {coef: {feat: 0 for feat in all_features} for coef in COEF_COLS}
    fold_count = 0

    for train_idx, test_idx in splitter.split(df_in):
        train_df = df_in.iloc[train_idx].reset_index(drop=True)
        X_train = build_symbolic_features(train_df, columns=all_features)

        for coef in COEF_COLS:
            y = train_df[coef].to_numpy(dtype=float)
            scaler = StandardScaler()
            Xs = scaler.fit_transform(X_train)
            model = LassoCV(cv=min(5, len(train_df)), random_state=fold_count + 1, max_iter=30000)
            model.fit(Xs, y)

            for feat, val in zip(all_features, model.coef_):
                if abs(val) > threshold:
                    stability[coef][feat] += 1

        fold_count += 1

    rows = []
    for coef in COEF_COLS:
        for feat in all_features:
            rows.append({
                "coefficient": coef,
                "term": feat,
                "frequency": stability[coef][feat] / fold_count,
                "count": stability[coef][feat],
                "folds": fold_count,
            })

    return pd.DataFrame(rows)

stability_df = term_stability_table(coef_df)
display(stability_df.sort_values("frequency", ascending=False).head(30))

In [ ]:
# Stability heatmap

stable_pivot = stability_df.pivot(index="coefficient", columns="term", values="frequency").fillna(0.0)

plt.figure(figsize=(max(12, 0.45 * stable_pivot.shape[1]), 4))
plt.imshow(stable_pivot.values, aspect="auto", vmin=0, vmax=1)
plt.yticks(range(len(stable_pivot.index)), stable_pivot.index)
plt.xticks(range(len(stable_pivot.columns)), stable_pivot.columns, rotation=45, ha="right")
plt.colorbar(label="selection frequency")
plt.title("Symbolic term stability across folds")
plt.tight_layout()
plt.show()

## Build pruned symbolic chart

Keep terms that are stable for at least one coefficient.

Default threshold:

\[
\text{frequency} \ge 0.5
\]

In [ ]:
STABILITY_THRESHOLD = 0.5

stable_terms_by_coef = {}
for coef in COEF_COLS:
    sub = stability_df[(stability_df["coefficient"] == coef) & (stability_df["frequency"] >= STABILITY_THRESHOLD)]
    stable_terms_by_coef[coef] = sub["term"].tolist()

stable_union_terms = sorted(set(sum(stable_terms_by_coef.values(), [])))

print("Stable union term count:", len(stable_union_terms))
print(stable_union_terms)

stable_terms_table = pd.DataFrame([
    {"coefficient": coef, "stable_terms": ", ".join(stable_terms_by_coef[coef]), "n_stable_terms": len(stable_terms_by_coef[coef])}
    for coef in COEF_COLS
])
display(stable_terms_table)

## Holdout evaluation

Compare:

1. Full symbolic chart
2. Pruned symbolic chart
3. Dense ridge chart

on harder blocks.

In [ ]:
def predict_sparse_models(train_df, test_df, feature_terms=None):
    X_train = build_symbolic_features(train_df, reduced_terms=feature_terms)
    X_test = build_symbolic_features(test_df, columns=X_train.columns, reduced_terms=feature_terms)

    Yhat = np.zeros((len(test_df), len(COEF_COLS)), dtype=float)

    for j, coef in enumerate(COEF_COLS):
        y_train = train_df[coef].to_numpy(dtype=float)
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_train)
        Xte = scaler.transform(X_test)
        model = LassoCV(cv=min(5, len(train_df)), random_state=42, max_iter=30000)
        model.fit(Xtr, y_train)
        Yhat[:, j] = model.predict(Xte)

    return Yhat

def predict_pruned_per_coef(train_df, test_df, terms_by_coef):
    Yhat = np.zeros((len(test_df), len(COEF_COLS)), dtype=float)

    for j, coef in enumerate(COEF_COLS):
        terms = terms_by_coef.get(coef, [])
        if len(terms) == 0:
            # intercept-only fallback
            Yhat[:, j] = train_df[coef].mean()
            continue

        X_train = build_symbolic_features(train_df, reduced_terms=terms)
        X_test = build_symbolic_features(test_df, columns=X_train.columns, reduced_terms=terms)

        y_train = train_df[coef].to_numpy(dtype=float)
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_train)
        Xte = scaler.transform(X_test)

        model = Ridge(alpha=1.0)
        model.fit(Xtr, y_train)
        Yhat[:, j] = model.predict(Xte)

    return Yhat

def predict_ridge(train_df, test_df):
    X_train = build_symbolic_features(train_df)
    X_test = build_symbolic_features(test_df, columns=X_train.columns)

    model = Ridge(alpha=1.0)
    model.fit(X_train, train_df[COEF_COLS].to_numpy(dtype=float))
    return model.predict(X_test)

hard_block_masks = {
    "k_extrapolate_high": coef_df["k"].astype(float) == 7.0,
    "k_extrapolate_low": coef_df["k"].astype(float) == 3.0,
    "system_holdout::entropy": coef_df["system"].astype(str) == "entropy",
    "system_holdout::unevenness": coef_df["system"].astype(str) == "unevenness",
    "forcing_holdout::condition_gap": coef_df["forcing_mode"].astype(str) == "condition_gap",
}

rows = []

for block_name, test_mask in hard_block_masks.items():
    train_df = coef_df.loc[~test_mask].reset_index(drop=True)
    test_df = coef_df.loc[test_mask].reset_index(drop=True)

    if len(test_df) == 0 or len(train_df) < 5:
        continue

    preds = {
        "full_symbolic": predict_sparse_models(train_df, test_df),
        "pruned_symbolic": predict_pruned_per_coef(train_df, test_df, stable_terms_by_coef),
        "ridge_chart": predict_ridge(train_df, test_df),
    }

    for i, rid in enumerate(test_df["regime_id"].tolist()):
        beta_true = test_df.loc[i, COEF_COLS].to_numpy(dtype=float)
        sub = regime_subsets[rid]
        y_emp = sub["predicted_flow"].to_numpy(dtype=float)

        for model_name, Yhat in preds.items():
            beta_hat = Yhat[i]
            pred_field = predict_with_beta(sub, beta_hat)
            rows.append({
                "block": block_name,
                "regime_id": rid,
                "model": model_name,
                "coef_rmse": math.sqrt(mean_squared_error(beta_true, beta_hat)),
                "field_rmse": math.sqrt(mean_squared_error(y_emp, pred_field)),
                "traj_rmse": trajectory_gap(sub, beta_true, beta_hat),
            })

eval_df = pd.DataFrame(rows)
display(eval_df.head())

In [ ]:
summary_df = eval_df.groupby(["block", "model"])[["coef_rmse", "field_rmse", "traj_rmse"]].mean().reset_index()
display(summary_df.sort_values(["block", "traj_rmse"]))

for metric in ["coef_rmse", "field_rmse", "traj_rmse"]:
    pivot = summary_df.pivot(index="block", columns="model", values=metric)
    ax = pivot.plot(kind="bar", figsize=(13, 5))
    ax.set_title(metric)
    ax.set_ylabel(metric)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## Regime-family rules

Extract mean coefficient templates by forcing and flow.

In [ ]:
family_rules = []

for family_cols in [["forcing_mode"], ["flow_mode"], ["forcing_mode", "flow_mode"], ["system"]]:
    grouped = coef_df.groupby(family_cols)[COEF_COLS].mean().reset_index()
    grouped["family"] = " × ".join(family_cols)
    family_rules.append(grouped)

family_rules_df = pd.concat(family_rules, ignore_index=True, sort=False)
display(family_rules_df.head(30))

In [ ]:
# Visualize family coefficient templates

for family_col in ["forcing_mode", "flow_mode", "system"]:
    temp = coef_df.groupby(family_col)[COEF_COLS].mean()
    plt.figure(figsize=(10, 4))
    plt.imshow(temp.values, aspect="auto", cmap="coolwarm")
    plt.yticks(range(len(temp.index)), temp.index)
    plt.xticks(range(len(COEF_COLS)), COEF_COLS, rotation=30, ha="right")
    plt.colorbar(label="mean coefficient")
    plt.title(f"Mean coefficient template by {family_col}")
    plt.tight_layout()
    plt.show()

## Final paper-ready equation block

Generate compact equations using only stable terms per coefficient.

In [ ]:
# Fit final pruned chart on all data with stable terms and print equations

final_equation_rows = []

for coef in COEF_COLS:
    terms = stable_terms_by_coef.get(coef, [])
    y = coef_df[coef].to_numpy(dtype=float)

    if len(terms) == 0:
        intercept = y.mean()
        eq = f"{coef} = {intercept:.5f}"
        final_equation_rows.append({"coefficient": coef, "n_terms": 0, "equation": eq})
        print(eq)
        print()
        continue

    X = build_symbolic_features(coef_df, reduced_terms=terms)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    model = Ridge(alpha=1.0)
    model.fit(Xs, y)

    # coefficients are in standardized feature units; still useful as model equations.
    active_terms = [(feat, val) for feat, val in zip(X.columns, model.coef_) if abs(val) > 1e-6]
    eq = format_equation(coef, active_terms, model.intercept_)
    final_equation_rows.append({"coefficient": coef, "n_terms": len(active_terms), "equation": eq})
    print(eq)
    print()

final_equations_df = pd.DataFrame(final_equation_rows)
display(final_equations_df)

## Final decision table

In [ ]:
decision_rows = []

for block, sub in summary_df.groupby("block"):
    best = sub.sort_values("traj_rmse").iloc[0]
    pruned = sub[sub["model"] == "pruned_symbolic"].iloc[0]
    full = sub[sub["model"] == "full_symbolic"].iloc[0]
    ridge = sub[sub["model"] == "ridge_chart"].iloc[0]

    pruned_to_best = pruned["traj_rmse"] / max(best["traj_rmse"], 1e-12)

    if best["model"] == "pruned_symbolic":
        verdict = "minimal symbolic chart wins"
    elif pruned_to_best <= 1.25:
        verdict = "minimal symbolic chart competitive"
    else:
        verdict = "full chart needed"

    decision_rows.append({
        "block": block,
        "best_model": best["model"],
        "best_traj_rmse": best["traj_rmse"],
        "pruned_traj_rmse": pruned["traj_rmse"],
        "full_traj_rmse": full["traj_rmse"],
        "ridge_traj_rmse": ridge["traj_rmse"],
        "pruned_to_best_ratio": pruned_to_best,
        "verdict": verdict,
    })

decision_df = pd.DataFrame(decision_rows)
display(decision_df)

## Working conclusion

Notebook 62 formalizes the static symbolic law as the final coordinate chart.

Interpretation guide:

- If `pruned_symbolic` is competitive, the final result is a minimal symbolic coordinate chart.
- If `full_symbolic` wins but `pruned_symbolic` fails, the chart exists but needs richer regime terms.
- If `ridge_chart` wins, symbolic sparsity is useful for interpretation but dense regularization is the better predictor.

Recommended next notebook:

**Notebook 63 — paper synthesis: governing field, symbolic chart, and universality boundary**